In [2]:
# Load the libraries required for model inference.

import numpy as np
import pandas as pd
import joblib

In [3]:
# Load the trained champion model.
model = joblib.load("../models/random_forest_log_model.pkl")

# Load the preprocessing pipeline used during training.
preprocessor = joblib.load("../models/preprocessor.pkl")

print("Model and preprocessor loaded successfully.")

Model and preprocessor loaded successfully.


In [5]:
# Load the cleaned dataset created during data exploration.
df = pd.read_csv("../data/processed/cleaned_transactions.csv")

# Remove columns that were not used as model features.
# PRICE_PER_SQM is excluded because it is derived from TRANS_VALUE
# and would cause target leakage.

X = df.drop(
    columns=["TRANS_VALUE", "PRICE_PER_SQM"]
)

# Take one property as a test input.
sample_property = X.iloc[[0]]

print(sample_property)

   GROUP_EN                                      PROCEDURE_EN IS_OFFPLAN_EN  \
0  Mortgage  Portfolio Mortgage Registration Pre-Registration      Off-Plan   

  IS_FREE_HOLD_EN     USAGE_EN       AREA_EN PROP_TYPE_EN PROP_SB_TYPE_EN  \
0       Free Hold  Residential  BUSINESS BAY         Unit            Flat   

   PROCEDURE_AREA  ACTUAL_AREA ROOMS_EN PARKING  \
0           46.17        46.17   Studio       1   

                       NEAREST_METRO_EN NEAREST_MALL_EN NEAREST_LANDMARK_EN  \
0  Buj Khalifa Dubai Mall Metro Station      Dubai Mall      Downtown Dubai   

      PROJECT_EN  YEAR  MONTH  DAY_OF_WEEK  
0  THE CRESTMARK  2026      7            2  


In [6]:
# Apply the saved preprocessing pipeline to the sample property.

sample_processed = preprocessor.transform(sample_property)

print("Processed shape:", sample_processed.shape)

Processed shape: (1, 16590)


In [7]:
# Predict the transaction value in log scale.
sample_prediction_log = model.predict(sample_processed)

# Convert the prediction back to AED.
sample_prediction = np.expm1(sample_prediction_log)

print(
    f"Predicted Transaction Value: AED {sample_prediction[0]:,.2f}"
)

Predicted Transaction Value: AED 930,940.96


In [8]:
def predict_transaction_value(property_data):
    # Convert the input dictionary into a one-row DataFrame.
    property_df = pd.DataFrame([property_data])

    # Apply the exact preprocessing pipeline used during training.
    property_processed = preprocessor.transform(property_df)

    # Predict the transaction value in log scale.
    prediction_log = model.predict(property_processed)

    # Convert the prediction back to AED.
    prediction_aed = np.expm1(prediction_log)[0]

    return prediction_aed

In [9]:
# Example property input.
# The column names must match the features used during training.

property_input = {
    "GROUP_EN": "Mortgage",
    "PROCEDURE_EN": "Portfolio Mortgage Registration Pre-Registration",
    "IS_OFFPLAN_EN": "Off-Plan",
    "IS_FREE_HOLD_EN": "Free Hold",
    "USAGE_EN": "Residential",
    "AREA_EN": "BUSINESS BAY",
    "PROP_TYPE_EN": "Unit",
    "PROP_SB_TYPE_EN": "Flat",
    "PROCEDURE_AREA": 46.17,
    "ACTUAL_AREA": 46.17,
    "ROOMS_EN": "Studio",
    "PARKING": "1",
    "NEAREST_METRO_EN": "Buj Khalifa Dubai Mall Metro Station",
    "NEAREST_MALL_EN": "Dubai Mall",
    "NEAREST_LANDMARK_EN": "Downtown Dubai",
    "PROJECT_EN": "THE CRESTMARK",
    "YEAR": 2026,
    "MONTH": 7,
    "DAY_OF_WEEK": 2
}

prediction = predict_transaction_value(property_input)

print(f"Predicted Transaction Value: AED {prediction:,.2f}")

Predicted Transaction Value: AED 930,940.96
